In [46]:
import numpy as np
import pandas as pd
import warnings
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.stats import norm

warnings.filterwarnings('ignore')

In [3]:
# Read portfolio CSV file
portfolio_df = pd.read_csv('./portfolio.csv', index_col=0)
portfolio_df.head()

,Weightage
Symbol,
ACC,0.008679
ASIANPAINT,0.011217
AWL,0.028318
BAJAJHFL,0.010478
BAJAJHIND,0.006915


In [4]:
# Download past 7 years of OHLC data for portfolio equities
end_dt = datetime.today()
start_dt = end_dt - timedelta(days=7*365)
chunk_size = 700
equities = list(portfolio_df.index)
all_equities_df = {}
benchmark_df = None

for equity in equities + ['^NSEI']:
    all_data = []
    loop_start_dt = start_dt
    while loop_start_dt < end_dt:
        loop_end_dt = loop_start_dt + timedelta(days=chunk_size)
        if loop_end_dt > end_dt:
            loop_end_dt = end_dt
        
        data = yf.download(
            f'{equity}.NS' if equity != '^NSEI' else equity,
            start=loop_start_dt.strftime('%Y-%m-%d'),
            end=loop_end_dt.strftime('%Y-%m-%d'),
            progress=False,
            auto_adjust=False,
        )
        if data.empty:
            print(f'No data received for {equity} between {loop_start_dt} and {loop_end_dt}')
        else:
            all_data.append(data)
        
        loop_start_dt = loop_end_dt
        
    # Combine all chunks
    df = pd.concat(all_data)
    df = df[~df.index.duplicated(keep='first')]  # Remove duplicate timestamps if any

    # Display the result
    df = df.droplevel(level='Ticker', axis=1)
    if equity == '^NSEI':
        benchmark_df = df
    else:
        all_equities_df[equity] = df



1 Failed download:
['AWL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-06-22 -> 2021-05-22) (Yahoo error = "Data doesn\'t exist for startDate = 1561141800, endDate = 1621621800")')


No data received for AWL between 2019-06-22 16:35:22.305560 and 2021-05-22 16:35:22.305560



1 Failed download:
['BAJAJHFL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-06-22 -> 2021-05-22) (Yahoo error = "Data doesn\'t exist for startDate = 1561141800, endDate = 1621621800")')


No data received for BAJAJHFL between 2019-06-22 16:35:22.305560 and 2021-05-22 16:35:22.305560



1 Failed download:
['BAJAJHFL.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-05-22 -> 2023-04-22) (Yahoo error = "Data doesn\'t exist for startDate = 1621621800, endDate = 1682101800")')


...

No data received for GATEWAY between 2019-06-22 16:35:22.305560 and 2021-05-22 16:35:22.305560



1 Failed download:
['MANYAVAR.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-06-22 -> 2021-05-22) (Yahoo error = "Data doesn\'t exist for startDate = 1561141800, endDate = 1621621800")')


No data received for MANYAVAR between 2019-06-22 16:35:22.305560 and 2021-05-22 16:35:22.305560



1 Failed download:
['SONACOMS.NS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-06-22 -> 2021-05-22) (Yahoo error = "Data doesn\'t exist for startDate = 1561141800, endDate = 1621621800")')


No data received for SONACOMS between 2019-06-22 16:35:22.305560 and 2021-05-22 16:35:22.305560


In [5]:
# Calculate returns for each column
benchmark_df['returns'] = benchmark_df['Adj Close'].pct_change()
benchmark_df.dropna(inplace=True)

for df in all_equities_df.values():
    df['returns'] = df['Adj Close'].pct_change()
    df.dropna(inplace=True)

In [ ]:
np.random.seed(100)
benchmark_start_dt = benchmark_df.index[0]

for equity, df in all_equities_df.items():
    print(equity)
    # Get benchmark data from equity start date
    equity_start_dt = df.index[0]
    bm_chunk_df = benchmark_df.loc[equity_start_dt:, ['returns']]
    
    # Remove indexes which are not there in other df
    if len(bm_chunk_df) > len(df):
        extra_indexes = set(bm_chunk_df.index) - set(df.index)
        bm_chunk_df.drop(extra_indexes, inplace=True)
    else:
        extra_indexes = set(df.index) - set(bm_chunk_df.index)
        df.drop(extra_indexes, inplace=True)
    
    if benchmark_start_dt >= equity_start_dt:
        continue

    # Get alpha and beta using np.polyfit method
    alpha, beta = np.polyfit(bm_chunk_df['returns'], df['returns'], 1)

    # Calculate modeled returns using alpha, beta for existing data frame
    modeled_returns = alpha + beta*bm_chunk_df

    # Calc residuals
    residuals = df['returns'] - modeled_returns['returns']

    # Calculate noise using residuals
    residuals_mean = residuals.mean()
    residuals_std = residuals.std()

    # Calculate historical data for the equity
    modeled_returns = alpha + beta*benchmark_df.loc[:equity_start_dt, ['returns']]
    error_term = np.random.normal(residuals_mean, residuals_std, len(modeled_returns))
    modeled_returns['returns'] = modeled_returns['returns'] + error_term
    
    df = pd.concat([modeled_returns, df])
    df = df[~df.index.duplicated(keep='last')]
    all_equities_df[equity] = df

AWL
BAJAJHFL
CLEAN
DOLATALGO
EASEMYTRIP
GATEWAY
HAPPSTMNDS
INDIGOPNTS
IRCTC
MANYAVAR
ROUTE
SBICARD
SONACOMS


In [7]:
returns_list = []

for equity, df in all_equities_df.items():
    equity_returns = df['returns']
    equity_returns.name = equity
    returns_list.append(equity_returns)

returns_df = pd.concat(returns_list, axis=1)
returns_df.head()

,ACC,ASIANPAINT,AWL,BAJAJHFL,BAJAJHIND,BANDHANBNK,BATAINDIA,BSOFT,CERA,CLEAN,...,TANLA,TATAELXSI,TCIEXP,TCS,TEAMLEASE,TTKPRESTIG,VAIBHAVGBL,VINATIORGA,WHIRLPOOL,WIPRO
Date,,,,,,,,,,,,,,,,,,,,,
2019-06-25,0.011024,-0.009483,-0.048608,-0.014606,-0.006536,-0.011613,0.005799,-0.017592,0.000371,0.024163,...,-0.049893,0.007633,0.003363,-0.003384,-0.008044,-0.004349,0.035150,0.002243,0.002899,0.004932
2019-06-26,0.018269,-0.002311,0.009559,-0.019481,-0.013158,0.013928,0.010860,-0.008953,0.019036,-0.003743,...,0.049512,0.009789,0.006464,-0.005997,-0.009170,-0.013654,0.010343,-0.007068,0.013058,0.002979
2019-06-27,-0.001232,0.001066,0.032088,-0.044908,0.026667,0.005140,0.012912,0.019198,-0.005488,-0.007229,...,0.049321,0.010848,0.028308,-0.000732,-0.001647,-0.001797,0.017452,-0.001400,0.014686,-0.014153
2019-06-28,-0.011923,-0.002387,-0.006981,0.002861,0.006494,0.001766,-0.000587,-0.043767,-0.008677,-0.014324,...,0.011580,0.010275,0.015885,-0.011254,0.002058,0.008484,0.034669,0.011192,0.004930,-0.005672
2019-07-01,0.000352,-0.003166,0.027312,-0.025605,0.019355,0.012529,0.000449,0.039397,-0.001626,0.016244,...,-0.030976,0.015821,-0.019432,0.005545,0.023933,-0.000459,0.027239,0.005288,0.026288,0.005347


## Calculating Parametric VaR and Expected Shortfall 

In [66]:
# Calculate mean and std dev of portfolio using historical data
columns = list(portfolio_df.index)
portfolio_returns = (returns_df[columns]*portfolio_df['Weightage']).sum(axis=1)
portfolio_returns = np.array(portfolio_returns)
portfolio_returns.sort()

daily_port_mean = portfolio_returns.mean()
daily_port_std = portfolio_returns.std()

print(daily_port_mean, daily_port_std)

0.000415203971747692 0.009626668724973602


In [79]:
# Assuming the returns of the portfolio follows normal distribution
# Let's calculate VaR with 95% confidence level

confidence_level = 0.95
significance_level = 1 - confidence_level

p_var_daily = norm.ppf(significance_level, daily_port_mean, daily_port_std)
p_cvar_daily = portfolio_returns[portfolio_returns < p_var_daily].mean()

p_var_ann = p_var_daily*np.sqrt(252)
p_cvar_ann = p_cvar_daily*np.sqrt(252)

print(f'Daily portfolio VaR and C-VaR are: {p_var_daily:.5f} and {p_cvar_daily:.5f}')
print(f'Annual portfolio VaR and C-VaR are: {p_var_ann:.5f} and {p_cvar_ann:.5f}')
# -0.1468057652152968

Daily portfolio VaR and C-VaR are: -0.01542 and -0.02495
Annual portfolio VaR and C-VaR are: -0.24477 and -0.39611


## Calculating Historical VaR and Expected Shortfall

In [80]:
confidence_level = 0.95
significance_level = 1 - confidence_level

h_var_daily = np.quantile(portfolio_returns, significance_level)
h_cvar_daily = portfolio_returns[portfolio_returns < h_var_daily].mean()

h_var_ann = h_var_daily*np.sqrt(252)
h_cvar_ann = h_cvar_daily*np.sqrt(252)

print(f'Daily portfolio VaR and C-VaR are: {h_var_daily:.5f} and {h_cvar_daily:.5f}')
print(f'Annual portfolio VaR and C-VaR are: {h_var_ann:.5f} and {h_cvar_ann:.5f}')

Daily portfolio VaR and C-VaR are: -0.01498 and -0.02428
Annual portfolio VaR and C-VaR are: -0.23785 and -0.38541
